# illust-asset-forge

上から順に実行する。各セルで書き換えるのは、先頭の変数（キャラクターの ID・枚数・表情名・画風名）だけ。

## 全体の流れ

- **準備**（1・2）: リポジトリとモデルを用意する
- **画風の見比べ**（3、任意）: 同じ顔を画風ごとに見る
- **顔**（4）: 作って、ボタンで採用する（中断した後は 5 で候補を開き直す）
- **表情**（6・7）: 範囲を赤枠で決めてから作り、ボタンで採用する
- **全身**（8）: 作って、ボタンで採用する
- **まとめ**（9）: zip にして Claude に渡す

## 1. 準備（リポジトリ・ライブラリ）

Drive をつなぎ、リポジトリを取り、ライブラリを入れる。Claude が設定を直して push した後は、**このセルだけ**を実行し直せば取り込める（モデルは読み込み直さない）。

In [ ]:
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")

# コピーして使うときは、この 4 つを書き換える
REPO_URL = "https://github.com/akabee0161/character-tactics.git"  # 取るリポジトリ
REPO_BRANCH = "develop"  # 取るブランチ
SUBDIR = "illust-asset-forge"  # リポジトリの中の forge のフォルダ（リポジトリの根なら空）
DRIVE_DIR = "/content/drive/MyDrive/character-tactics/illust-asset-forge"  # 候補を置く Drive の場所（作品ごとに分ける）

CLONE = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
ROOT = os.path.normpath(os.path.join(CLONE, SUBDIR))


def git(*args, capture=False):
    return subprocess.run(["git", "-C", CLONE, *args], check=True, capture_output=capture, text=True)


def same_repo(a, b):
    return a.strip().rstrip("/").removesuffix(".git") == b.strip().rstrip("/").removesuffix(".git")


if not os.path.isdir(CLONE):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, CLONE], check=True)
else:
    origin = git("remote", "get-url", "origin", capture=True).stdout
    if not same_repo(origin, REPO_URL):
        raise RuntimeError(f"{CLONE} は別のリポジトリ（{origin.strip()}）の clone です。消してから実行し直してください: !rm -rf {CLONE}")
    git("fetch", "origin")
    git("checkout", REPO_BRANCH)
    git("pull")

# `!pip` は失敗しても例外にならず、古い依存のまま先へ進む。失敗したらセルを止める
for args in (["-r", f"{ROOT}/requirements-colab.txt"], ["-e", ROOT, "--no-deps"]):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

os.chdir(ROOT)
# pip install -e は、動いているカーネルの import には効かない。src を直接足す
SRC = os.path.join(ROOT, "src")
# 同じカーネルで ROOT を切り替えたときに、前の src が先に読まれないよう、今の src を先頭へ移す
while SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)
# 実行し直したときに古いコードのままにならないよう、読み込み済みの artgen を捨てる
for name in [m for m in sys.modules if m == "artgen" or m.startswith("artgen.")]:
    del sys.modules[name]
from artgen import forge as F
from artgen import picker as P
if "pipe" in globals():
    forge = F.attach(pipe, ".", base_dir=DRIVE_DIR)

## 2. モデルの読み込み

`pipe` が無いときだけ読み込む（2 回目以降は何もしない）。メモリ不足で止まったときだけ、`F.load_pipeline(low_memory=True)` にする（遅くなる）。

**終わったら** → 次へ。

In [ ]:
if "pipe" not in globals():
    pipe = F.load_pipeline()
forge = F.attach(pipe, ".", base_dir=DRIVE_DIR)

## 3. 画風の見比べ（任意）

`CHAR` の顔を、画風ごとに 2 枚ずつ作って並べる。気に入った画風名を、後ろの `STYLE` に書く。

In [ ]:
CHAR = "knight"
paths = forge.compare_styles(CHAR, seeds=None, n=2)
P.show_styles(paths)
# その場で画風を書いて混ぜる例（YAML を書かずに試せる。気に入ったら Claude に渡して styles/ に入れてもらう）
# paths = forge.compare_styles(CHAR, styles=["<画風名>", {"name": "my-style", "prompt": ["flat color", "thin lineart"], "negative": ["gradient"]}], n=2)

## 4. 顔

`CHARS` に作るキャラクターを並べる。1 枚できるたびに表示される（1 枚 20 秒ほど）。調整中は `n=2` など少なくし、一気に作るときは `n=16` などにして、後から下の「候補を開き直す」で選ぶ。`STYLE = None` なら `defaults.yaml` のデフォルトの画風（`character.yaml` に `style` があればそれ）。

できたら、出てくる画面で採用する顔のボタンを押す。

In [ ]:
CHARS = ["knight", "mage", "schoolboy", "schoolgirl"]
STYLE = None
paths = forge.generate(CHARS, "face", n=4, style=STYLE, on_image=F.show_one)
P.pick(forge, paths)

## 5. 候補を開き直す

中断・切断の後や、後から選ぶときに使う。候補は Drive の `runs/<キャラクター>/<種類>/<日時>/` に入っている。`char` と `kind`（`face`・`fullbody` など）を書くと、そのキャラクター・種類の最新の候補を開く。`run_id` に日時のフォルダ名を足すと、その日時の候補を開く。`char`・`kind` を書かなければ、最新の日時の全部を開く。古い `runs/<日時>/` のフォルダは `char`・`kind` では絞れないので、`P.pick(forge, run_id="<日時>")` だけで開く（`char`・`kind` は書かない）。

In [ ]:
CHAR = "knight"
KIND = "face"
P.pick(forge, char=CHAR, kind=KIND)
# P.pick(forge, char=CHAR, kind=KIND, run_id="20261122_093000")  # 日時を決めて開く
# P.pick(forge, run_id=None)  # 最新の日時の全部
# P.pick(forge, run_id="<古い日時>")  # 古い runs/<日時>/ のフォルダ（char・kind は書かない）

## 6. 表情の範囲

採用した顔に、目と口の範囲（赤枠）をスライダーで合わせて保存する。表情を作る前に、キャラクターごとに 1 回やる。

In [ ]:
CHAR = "knight"
P.edit_box(forge, CHAR)

## 7. 表情

採用した顔と決めた範囲から作る（パスは書かない）。表情名は `character.yaml` の `expressions` のキー（`hurt`・`shout` など）。ほかの表情は `EXPRESSION` を書き換えて実行し直す。

In [ ]:
CHAR = "knight"
EXPRESSION = "hurt"
paths = forge.inpaint(CHAR, EXPRESSION, n=4, on_image=F.show_one)
P.pick(forge, paths)

## 8. 全身（参考資料）

顔と同じく、作ってからボタンで採用する。見るのは、服の種類・色・持ち物と、髪の色・髪型が顔と合っているか。

In [ ]:
CHARS = ["knight", "mage", "schoolboy", "schoolgirl"]
paths = forge.generate(CHARS, "fullbody", n=4, on_image=F.show_one)
P.pick(forge, paths)

## 9. まとめ

採用したものを zip にして、ダウンロードする。この zip を Claude とのチャットに添付する。

In [ ]:
z = forge.pack()
print(z)
try:
    from google.colab import files
    files.download(str(z))
except Exception as e:
    print(f"ダウンロードできなかった（Drive の packs から取る）: {e}")

## 試すときの書き方

- 足すタグ・消すタグ: `forge.generate(CHARS, "face", extra="scar on cheek", remove="stubble")`（ネガティブは `extra_negative=`・`remove_negative=`）。`remove` は設定のタグにだけ効く。`extra` に設定と同じタグを重み付きで書くと、設定のタグが置き換わる（例: `extra="(stubble:1.3)"`）
- シードを固定: `seeds=[123, 456]`（渡すと `n` は無視される）
- 画風: `style="<画風名>"`
- その場で画風を書く例: `style={"name": "my-style", "prompt": ["flat color", "thin lineart"], "negative": ["gradient"]}`（`compare_styles(styles=[...])` の要素にも書ける。画風名と混ぜてよい）。気に入ったら、採用した絵の zip（または、この辞書）を Claude に渡すと、Claude が `styles/my-style.yaml` にする
- guidance・steps（inpaint は `strength` も）: `guidance=6`、`steps=30`
- 選ぶ画面を使わずに採用する: `forge.adopt("<ファイル名>", flip=True)`（`flip=True` で左右反転して採用）
- 画像を並べて見る: `F.show(paths)`
- **YAML は Colab で書き換えない**（次の pull でぶつかる）。上書きの内容は PNG の条件（`overrides`）に残るので、Claude が設定に戻す。